In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*pandas only supports SQLAlchemy.*")
warnings.filterwarnings("ignore", category=RuntimeWarning)

import pymysql
import pandas as pd
import ast
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

def get_db_connection():
    """建立 StarRocks 数据库连接"""
    return pymysql.connect(
        host='192.168.144.101',
        port=9030,
        user='hadoop',
        password='hadoop',
        charset='utf8'
    )

def fetch_cities():
    """获取 ADS 表中有数据的城市列表"""
    conn = get_db_connection()
    query = """
    SELECT DISTINCT city 
    FROM iceberg_catalog.ershoufang.ads_top10_attention_houses_daily 
    WHERE city IS NOT NULL AND city != ''
    ORDER BY city
    """
    try:
        df = pd.read_sql(query, conn)
        return df['city'].tolist()
    finally:
        conn.close()

def format_tags(val):
    if not val or pd.isna(val) or val == '[]':
        return '-'
    if isinstance(val, list):
        return ' · '.join([str(v).strip() for v in val if str(v).strip()])
    if isinstance(val, str):
        try:
            lst = ast.literal_eval(val)
            if isinstance(lst, list):
                return ' · '.join([str(v).strip() for v in lst if str(v).strip()])
        except:
            return val
    return val

In [ ]:
def fetch_top10_houses_data(city_name):
    conn = get_db_connection()
    query = f"""
    WITH LatestDate AS (
        SELECT MAX(ds) as max_ds 
        FROM iceberg_catalog.ershoufang.ads_top10_attention_houses_daily 
        WHERE city = '{city_name}'
    )
    SELECT 
        rank as `排名`,
        title,
        house_url,
        community as `小区名称`,
        total_price as `总价(万)`,
        unit_price as `单价(元/㎡)`,
        area_sqm as `面积(㎡)`,
        attention_count as `关注人数`,
        visit_30_days as `30天带看`,
        tags as `房源标签`
    FROM iceberg_catalog.ershoufang.ads_top10_attention_houses_daily
    WHERE city = '{city_name}' 
      AND ds = (SELECT max_ds FROM LatestDate)
    ORDER BY `排名` ASC
    LIMIT 10
    """
    try:
        df = pd.read_sql(query, conn)
        return df
    finally:
        conn.close()

In [ ]:
def render_styled_table(city):
    """渲染带微型条形图的 HTML 富文本表格"""
    df = fetch_top10_houses_data(city)
    
    if df.empty:
        display(HTML(f"<div style='padding: 20px; font-size: 16px; color: #7F8C8D;'>{city} 暂无有效的高关注房源榜单数据。</div>"))
        return
    # 格式化标签
    df['房源标签'] = df['房源标签'].apply(format_tags)
    
    # 拼装可点击的超链接标题
    df['房源标题'] = df.apply(
        lambda row: f'<a href="{row["house_url"]}" target="_blank" style="color: #2980B9; text-decoration: none; font-weight: bold;">{row["title"]}</a>', 
        axis=1
    )
    
    # 调整列顺序并移除原始的 title 和 house_url
    display_cols = ['排名', '房源标题', '小区名称', '总价(万)', '单价(元/㎡)', '面积(㎡)', '关注人数', '30天带看', '房源标签']
    df = df[display_cols]

    # --- DataFrame 样式注入 ---
    styled_df = (df.style
        .hide(axis="index") 
        .bar(subset=['关注人数'], color='#F5B041', vmin=0, height=60, width=90) # 阳光橙
        .bar(subset=['30天带看'], color='#5DADE2', vmin=0, height=60, width=90) # 宁静蓝
        .bar(subset=['总价(万)'], color='#A9DFBF', vmin=0, height=60, width=90) # 薄荷绿
        # 数值格式化，加入千分位
        .format({
            '总价(万)': '{:.1f}',
            '单价(元/㎡)': '{:,.0f}',
            '面积(㎡)': '{:.2f}',
            '关注人数': '{:,} 人',
            '30天带看': '{:,} 次'
        })
        # 全局居中对齐与字体设置
        .set_properties(**{
            'text-align': 'center', 
            'vertical-align': 'middle',
            'font-family': 'SimHei, Microsoft YaHei, sans-serif',
            'font-size': '13px',
            'padding': '12px'
        })
        # 头部表头美化
        .set_table_styles([
            {'selector': 'th', 'props': [
                ('background-color', '#34495E'), 
                ('color', 'white'), 
                ('font-size', '14px'),
                ('text-align', 'center'),
                ('padding', '12px')
            ]},
            {'selector': 'tr:hover', 'props': [('background-color', '#F4F6F6')]},
            # 允许房源标题栏左对齐并稍微加宽
            {'selector': 'td:nth-child(2)', 'props': [('text-align', 'left'), ('min-width', '250px')]} 
        ])
    )
    
    # 输出带有标题的 HTML
    html_out = f"""
    <div style="margin-bottom: 20px;">
        <h3 style="color: #2C3E50; font-family: SimHei, sans-serif;">🏙️ [{city}] 每日关注度最高二手房源榜单 (TOP 10)</h3>
        <p style="color: #7F8C8D; font-size: 13px; font-style: italic;">视觉引导：单元格内的颜色条代表该指标在榜单中的相对大小。点击标题可直接访问房源详情。</p>
    </div>
    {styled_df.to_html()}
    """
    display(HTML(html_out))

In [4]:
def render_interactive_dashboard():
    """构建交互式下拉组件"""
    cities = fetch_cities()
    if not cities:
        print("未获取到城市列表，请检查数据。")
        return
    
    default_city = '北京' if '北京' in cities else cities[0]
    
    city_dropdown = widgets.Dropdown(
        options=cities,
        value=default_city,
        description='📍 分析城市:',
        layout={'width': '250px'}
    )
    
    plot_output = widgets.Output()

    def on_city_change(change):
        if change['type'] == 'change' and change['name'] == 'value':
            with plot_output:
                clear_output(wait=True)
                render_styled_table(change['new'])

    city_dropdown.observe(on_city_change)

    display(city_dropdown)
    display(plot_output)
    
    with plot_output:
        render_styled_table(default_city)

In [5]:
if __name__ == "__main__":
    render_interactive_dashboard()

Dropdown(description='📍 分析城市:', index=1, layout=Layout(width='250px'), options=('上海', '北京', '南京', '天津', '太原', …

Output()